# Training and Core ML export — sanitized historical notebook

This edited copy preserves the training and export code from retained records. **It was not executed during review or publication.** All original outputs, execution counts, and notebook metadata were removed. The original credential literal was replaced with the `ROBOFLOW_API_KEY` environment variable; storage paths were made relative and contextual comments were corrected. Do not put a credential in a saved notebook.

## Historical evidence from the original saved outputs

- A 3-epoch smoke run precedes a separate completed 50-epoch YOLO11n run (640 pixels, batch 64, patience 50).
- Dataset: Airborne-Object-Detection-4-AOD4 version 6; classes: airplane, bird, drone, helicopter; configuration declares CC BY 4.0.
- Subsequent notebook validation: 4,514 images, 6,369 instances; mAP50 **0.9651819355211275**, mAP50–95 **0.66663031065507**. These are historical four-class aggregate validation results, not drone-specific accuracy or independently reproduced measurements.
- The original output reports completion of a Core ML export with NMS enabled. Exact equivalence to the separate iOS asset and successful device inference remain unverified.
- The checkpoint stores separate metrics: mAP50 0.96511 and mAP50–95 0.66650. The records must not be silently merged.

## Reproduction limits

The original dependency installation was unpinned; its outputs identify Ultralytics 8.4.87. The retained notebook alone is not a locked reproducible environment. The `%uv` installation magic depends on the notebook host. Dataset access requires your own authorized credential. Plot and checkpoint listings in old outputs do not prove that every referenced artifact was recovered. Code below may train, download, copy, or export when manually executed; publication itself performed none of these actions.

See the repository model card and `NOTEBOOK_REVIEW.md` for attribution, licensing and provenance qualifications.



In [ ]:
%uv pip -q install ultralytics roboflow
import torch
from ultralytics import YOLO
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')


In [ ]:
from roboflow import Roboflow
import os
os.makedirs('dataset_workspace', exist_ok=True)
os.chdir('dataset_workspace')
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("airborne-object-detection").project("airborne-object-detection-4-aod4")
dataset = project.version(6).download("yolov11")
print("downloaded to:", dataset.location)   # note this path


In [ ]:
import os
dpath = dataset.location            # e.g. Airborne-Object-Detection-4-AOD4-6
print(open(os.path.join(dpath, 'data.yaml')).read())


In [ ]:
from ultralytics import YOLO
import os
YOLO('yolo11n.pt').train(
    data=os.path.join(dataset.location, 'data.yaml'),
    epochs=3, imgsz=640, batch=64, name='aod4_check', plots=True,
)


In [ ]:
from ultralytics import YOLO
import shutil, os
results = YOLO('yolo11n.pt').train(
    data=os.path.join(dataset.location, 'data.yaml'),
    epochs=50, imgsz=640, batch=64, name='aod4_v1',
    patience=50,
    plots=True,
)
os.makedirs('artifacts', exist_ok=True)
shutil.copy('runs/detect/aod4_v1/weights/best.pt', 'artifacts/best_aod4.pt')
print('\n===== DONE =====')
print('mAP50:', results.box.map50, '| mAP50-95:', results.box.map)
print('saved to artifacts/best_aod4.pt')


In [ ]:
import shutil, os

os.makedirs('artifacts/aod4_run', exist_ok=True)

# the trained weights (both best and last)
shutil.copy('runs/detect/aod4_v1/weights/best.pt', 'artifacts/aod4_run/best.pt')
shutil.copy('runs/detect/aod4_v1/weights/last.pt', 'artifacts/aod4_run/last.pt')

# the training results/plots/metrics (great for the HF model card + your records)
for f in ['results.csv', 'results.png', 'confusion_matrix.png',
          'BoxPR_curve.png', 'BoxF1_curve.png', 'args.yaml',
          'val_batch0_pred.jpg', 'labels.jpg']:
    src = f'runs/detect/aod4_v1/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'artifacts/aod4_run/{f}')

print('saved to artifacts/aod4_run:')
print(os.listdir('artifacts/aod4_run'))


In [ ]:
from ultralytics import YOLO
m = YOLO('artifacts/aod4_run/best.pt')
# Historical export setting; successful device inference was not established.
out = m.export(format='coreml', nms=True, imgsz=640)
print('exported:', out)

# retain the exported package in the artifacts directory
import shutil
shutil.move(out, 'artifacts/aod4_run/best.mlpackage')
print('saved mlpackage to artifacts/aod4_run/best.mlpackage')
